# Check API

End-to-end checks for `OpsLookup.Api` and the LangGraph service.

**Two fixes applied to the previous version of this notebook**

1. `asyncio.run(main())` cannot be used here. Jupyter already runs an event loop, so
   `asyncio.run()` raises `RuntimeError: cannot be called from a running event loop`.
   Every async test is now `await`ed at top level, which ipykernel supports.
2. The kernel was system Python 3.10, which does not have `langgraph` installed. The
   project venv is Python 3.14 and now has `ipykernel`. Select the **`opslookup`**
   kernel from the notebook toolbar before running anything.

**Before running**

- Start the API: `cd src/OpsLookup.Api` then `dotnet run`
- `API_TOKEN` is optional. This notebook mints its own token from `/auth/dev-token`.

In [60]:
import asyncio
import json
import os
import sys

import httpx
from dotenv import load_dotenv

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END

from graph.build import build_graph, route_after_llm
from tools.api_client import LookupApiClient
from tools.contracts import build_tools

load_dotenv()

print(f"python      {sys.version.split()[0]}")
print(f"interpreter {sys.executable}")
print(f"LONGCAT key  {'set' if os.getenv('LONGCAT_API_KEY') else 'NOT set (real-model test will skip)'}")

python      3.10.5
interpreter C:\Users\saad\AppData\Local\Programs\Python\Python310\python.exe
LONGCAT key  set


In [61]:
RESULTS = []


def check(label, condition, detail=""):
    """Record a check. Does not raise, so one failure does not hide the rest."""
    RESULTS.append((label, bool(condition), detail))
    mark = "PASS" if condition else "FAIL"
    line = f"  [{mark}] {label}"
    if detail:
        line += f"  ({detail})"
    print(line)
    return bool(condition)


def heading(text):
    print(f"\n{text}")


def summary():
    failed = [label for label, ok, _ in RESULTS if not ok]
    print("\n" + "=" * 66)
    print(f"{len(RESULTS) - len(failed)}/{len(RESULTS)} checks passed")
    if failed:
        print("\nFailed:")
        for label in failed:
            print(f"  - {label}")
    print("=" * 66)
    return not failed

## 1. The .NET API

Auth, the lookup itself, the party-name filter, social ID validation, and the
401 path.

In [57]:
load_dotenv()

True

In [59]:
os.getenv("API_BASE_URL").rstrip("/")

'https://localhost:7163'

In [62]:
API_TOKEN = None


async def test_dotnet_api():
    heading("1. OpsLookup.Api")

    base_url = os.getenv("API_BASE_URL", "http://localhost:5250").rstrip("/")
    print(f"     API_BASE_URL = {base_url}")

    try:
        return await _run_api_checks(base_url)
    except httpx.ConnectError:
        check("API is reachable", False, f"cannot connect to {base_url}")
        print()
        print("     Fix one of these:")
        print(f"       - start it:        cd ../OpsLookup.Api && dotnet run")
        print(f"       - or set API_BASE_URL in .env to the port it actually listens on")
        print(f"         launchSettings.json uses http://localhost:5250 and https://localhost:7163")
        print()
        print("     If you use the https URL, the ASP.NET Core dev certificate is")
        print("     self-signed and httpx will refuse it. Either point .env at the http")
        print("     URL, or export SSL_CERT_FILE with the dev cert exported to PEM.")
        return None
    except httpx.ConnectError as exc:
        raise


async def _run_api_checks(base_url):
    async with httpx.AsyncClient(base_url=base_url, timeout=20.0) as http:
        response = await http.get("/health")
        if not check("GET /health is 200", response.status_code == 200, f"got {response.status_code}"):
            return None

        response = await http.post("/auth/dev-token", params={"operatorId": "notebook"})
        check("POST /auth/dev-token is 200", response.status_code == 200, f"got {response.status_code}")
        token = response.json().get("accessToken")
        if not check("response contains accessToken", bool(token)):
            return None
        auth = {"Authorization": f"Bearer {token}"}

        response = await http.get("/api/contracts", params={"socialId": 1234567890}, headers=auth)
        check("GET /api/contracts is 200", response.status_code == 200, f"got {response.status_code}")
        rows = response.json()
        check("returns at least one row", len(rows) >= 1, f"{len(rows)} rows")

        if rows:
            expected = {"contractCode", "socialId", "partyName", "contractName"}
            check(
                "row has exactly the four expected fields",
                set(rows[0]) == expected,
                str(sorted(rows[0])),
            )
            print(f"     first row: {rows[0]}")

        response = await http.get(
            "/api/contracts", params={"socialId": 1234567890, "name": "karimi"}, headers=auth
        )
        check(
            "name filter excludes non-matching parties",
            response.json() == [],
            f"{len(response.json())} rows",
        )

        response = await http.get("/api/contracts", params={"socialId": 2111111111}, headers=auth)
        check("single-row lookup is 200", response.status_code == 200, f"got {response.status_code}")

        response = await http.get("/api/contracts", params={"socialId": 1999999999}, headers=auth)
        check(
            "valid but unknown socialId returns empty array, not an error",
            response.status_code == 200 and response.json() == [],
            f"HTTP {response.status_code}",
        )

        response = await http.get("/api/contracts", params={"socialId": 3456789012}, headers=auth)
        check("socialId starting with 3 is rejected 400", response.status_code == 400, f"got {response.status_code}")

        response = await http.get("/api/contracts", params={"socialId": 12345}, headers=auth)
        check("too-short socialId is rejected 400", response.status_code == 400, f"got {response.status_code}")

        response = await http.get("/api/contracts", params={"socialId": 1234567890})
        check("no bearer token is 401", response.status_code == 401, f"got {response.status_code}")

        response = await http.get("/api/contracts", params={"socialId": 1234567890}, headers=auth)
        print("\n--- SQL the API generated ---")
        print(f"     rows returned: {len(response.json())}")

    return token


API_TOKEN = await test_dotnet_api()


1. OpsLookup.Api
     API_BASE_URL = https://localhost:7163
  [FAIL] API is reachable  (cannot connect to https://localhost:7163)

     Fix one of these:
       - start it:        cd ../OpsLookup.Api && dotnet run
       - or set API_BASE_URL in .env to the port it actually listens on
         launchSettings.json uses http://localhost:5250 and https://localhost:7163

     If you use the https URL, the ASP.NET Core dev certificate is
     self-signed and httpx will refuse it. Either point .env at the http
     URL, or export SSL_CERT_FILE with the dev cert exported to PEM.


## 2. The graph loop

Runs the real agent loop against the real API using a stubbed model, so these
checks need no `LONGCAT_API_KEY`.

- a tool call routes to `action`, not `end`
- the API result reaches the next `llm` node
- plain text routes to `end`
- thread state survives across separate invocations on one `thread_id`

In [46]:
class StubLlm:
    """Emits one tool call, then a text answer once it has seen the result."""

    def __init__(self, social_id: int) -> None:
        self.social_id = social_id

    async def ainvoke(self, messages):
        if not any(isinstance(m, ToolMessage) for m in messages):
            return AIMessage(
                content="",
                tool_calls=[{
                    "name": "get_contract",
                    "args": {"social_id": self.social_id},
                    "id": "call_1",
                }],
            )
        payload = next(
            m.content for m in messages
            if isinstance(m, ToolMessage) and m.tool_call_id == "call_1"
        )
        return AIMessage(content=f"Tool returned {len(payload)} chars")


class StubAskingLlm:
    """Emits no tool call, so the graph must go straight to `end`."""

    async def ainvoke(self, messages):
        return AIMessage(content="Which social ID would you like me to look up?")

In [36]:
def tool_content(result) -> str:
    return next(m.content for m in result["messages"] if isinstance(m, ToolMessage))

In [37]:
async def test_graph():
    heading("2. LangGraph loop")

    if not API_TOKEN:
        print("     Skipped: no token from section 1.")
        return

    client = LookupApiClient(
        os.getenv("API_BASE_URL", "http://localhost:5250").rstrip("/"),
        API_TOKEN,
        "notebook",
    )
    tools = build_tools(client)

    try:
        # --- tool call -> action -> llm -> end ---------------------------------
        graph = build_graph(StubLlm(1234567890), tools, InMemorySaver())
        result = await graph.ainvoke(
            {"messages": [("user", "check social ID 1234567890")]},
            {"configurable": {"thread_id": "nb-1"}, "recursion_limit": 10},
        )
        kinds = [type(m).__name__ for m in result["messages"]]
        print(f"     message kinds: {kinds}")

        payload = tool_content(result)
        print(f"     tool returned: {payload[:150]}")

        check("action node ran (ToolMessage present)", "ToolMessage" in kinds)
        check("llm answered without another tool call", not result["messages"][-1].tool_calls)
        check("tool result carried real contract rows", "Annual Maintenance Agreement" in payload)
        check("tool result is not an error", '"error"' not in payload)

        # --- plain text -> end --------------------------------------------------
        graph2 = build_graph(StubAskingLlm(), tools, InMemorySaver())
        result2 = await graph2.ainvoke(
            {"messages": [("user", "check something")]},
            {"configurable": {"thread_id": "nb-2"}, "recursion_limit": 10},
        )
        check("clarifying turn is one reply", len(result2["messages"]) == 2, f"{len(result2['messages'])} messages")
        check("clarifying turn called no tool", not any(isinstance(m, ToolMessage) for m in result2["messages"]))

        # --- thread continuity --------------------------------------------------
        config = {"configurable": {"thread_id": "nb-3"}, "recursion_limit": 10}
        await graph.ainvoke({"messages": [("user", "first message")]}, config)
        result3 = await graph.ainvoke({"messages": [("user", "second message")]}, config)
        human = [m.content for m in result3["messages"] if isinstance(m, HumanMessage)]
        print(f"     human messages on thread: {human}")
        check("both turns retained on the thread", len(human) == 2, f"{len(human)} human messages")
        check("earlier turn still in history", human[0] == "first message")
        check("turn 1 tool result still in history", any(isinstance(m, ToolMessage) for m in result3["messages"]))
    finally:
        await client.aclose()


await test_graph()


2. LangGraph loop
     Skipped: no token from section 1.


In [38]:
heading("3. Routing helper")

calling = AIMessage(content="", tool_calls=[{"name": "x", "args": {}, "id": "1"}])
check("tool_calls routes to action", route_after_llm({"messages": [calling]}) == "action")
check("plain text routes to END", route_after_llm({"messages": [AIMessage(content="hi")]}) == END)


3. Routing helper
  [PASS] tool_calls routes to action
  [PASS] plain text routes to END


True

In [39]:
summary()


2/3 checks passed

Failed:
  - API is reachable


False

## 4. Real model, optional

Runs the same loop against LongCat for real. Skipped automatically when
`LONGCAT_API_KEY` is unset, so it does not affect the checks above.

If you want the natural-language behaviour rather than the stub, start the graph
service and call `/chat` instead:

```powershell
cd src\OpsLookup.Graph
.\.venv\Scripts\uvicorn api:app --port 8000
```

In [40]:
d=os.getenv('LONGCAT_API_KEY').
d

SyntaxError: invalid syntax (622734673.py, line 1)

In [42]:
heading("4. Real model (optional)")

if not os.getenv("LONGCAT_API_KEY"):
    print("     Skipped: LONGCAT_API_KEY is not set.")
elif not API_TOKEN:
    print("     Skipped: no token from section 1.")
else:
    from llm import build_llm
    from config import load_settings

    settings = load_settings()
    client = LookupApiClient(settings.api_base_url, API_TOKEN, "notebook")
    tools = build_tools(client)

    try:
        graph = build_graph(build_llm(settings, tools), tools, InMemorySaver())
        result = await graph.ainvoke(
            {"messages": [("user", "Can you check what contracts are held for social ID 1234567890?")]},
            {"configurable": {"thread_id": "nb-real"}, "recursion_limit": 10},
        )
        print(f"     reply: {result['messages'][-1].content}")
        check(
            "real model reached a final answer",
            not result["messages"][-1].tool_calls,
        )
    except Exception as exc:
        check("real model run completed", False, f"{type(exc).__name__}: {exc}")
    finally:
        await client.aclose()


4. Real model (optional)
     Skipped: LONGCAT_API_KEY is not set.
